In [0]:
from pyspark.sql.functions import current_timestamp

VOLUME_PATH = "/Volumes/olist_ecom/source/source_volume"

files = {
    "olist_orders_dataset.csv": "bronze_orders",
    "olist_customers_dataset.csv": "bronze_customers",
    "olist_geolocation_dataset.csv": "bronze_geolocation",
    "olist_order_items_dataset.csv": "bronze_order_items",
    "olist_order_payments_dataset.csv": "bronze_order_payments",
    "olist_order_reviews_dataset.csv": "bronze_order_reviews",
    "olist_products_dataset.csv": "bronze_products",
    "olist_sellers_dataset.csv": "bronze_sellers",
    "product_category_name_translation.csv": "bronze_category_translation",
}

for file_name, table_name in files.items():
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .csv(f"{VOLUME_PATH}/{file_name}")
          .withColumn("ingested_at", current_timestamp()))
    df.write.format("delta").mode("overwrite").saveAsTable(f"olist_ecom.bronze.{table_name}")
    print(f"{table_name}: {df.count()} rows")

In [0]:
spark.table("olist_ecom.bronze.bronze_order_reviews") \
     .groupBy("review_score").count().orderBy("review_score").show()

In [0]:
spark.sql("SELECT COUNT(*) FROM workspace.default.bronze_orders").show()